# Uczenie głębokie 


## laboratorium: transformer w keras

Najważniejsze koncepcje :
-   Atencja
-   Kodowanie pozycyjne
-   Transformer

## Baseline z poprzednich zajęć

In [1]:
import numpy as np
import os
os.environ["KERAS_BACKEND"] = "jax"
import keras

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

fname = "Star.Trek.Picard.S01E01.PROPER.WEBRip.x264-ION10.srt"
fname = "./Star.Trek.Picard.S01E01.PROPER.WEBRip.x264-ION10.srt"


with open(fname, "r", encoding="cp1250") as f:
    txt = f.read()
txts = txt.split("\n\n")


def parse(txt):
    subtxt = txt.split("\n")
    return subtxt[:2] + ["\n".join(subtxt[2:])]


df = pd.DataFrame([parse(txt) for txt in txts])
df.columns = ["index", "czas", "tekst"]
df = df.set_index("index")
df[["czas_start", "czas_stop"]] = df.czas.str.split("-->", expand=True)
df = df.drop("czas", axis=1)

df.czas_start = pd.to_timedelta(df.czas_start.str.replace(",", ".")) / pd.to_timedelta(
    "1s"
)
df.czas_stop = pd.to_timedelta(df.czas_stop.str.replace(",", ".")) / pd.to_timedelta(
    "1s"
)
df["n"] = df.tekst.str.findall("\n").apply(len) + 1


df["ns"] = df["n"].astype(str)
df["multiline"] = df["n"] > 1
df["n"] = pd.Categorical(df["n"], categories=[1, 2])

df["l"] = df.tekst.str.len()
df["t"] = df.czas_stop - df.czas_start
df

korpus = ''.join(df['tekst'])
len(korpus)


15623

In [3]:
dictionary = list(set(korpus))
dictionary

korpus_tokeny = [dictionary.index(c) for c in korpus]

print(''.join(dictionary))

ódrPK3ZzŃśAęuĘl0BF,łN4Mx„2tODŁmźć
Soc:pWX?wJGHLYR”gI.- ean7UET!bżkŚÂfiń9ĆCŹj16hyŻ5są


In [4]:
import keras
import tensorflow as tf
from IPython.display import display, Markdown
context = 128

ds = keras.utils.timeseries_dataset_from_array(korpus_tokeny, None, sequence_length=context, batch_size=64)

def splitfn(x):
    return x[...,:-1], x[...,-1]

ds = ds.map(splitfn)

for x,y in ds:
    break

print(x,y)
emb = keras.layers.Embedding(len(dictionary), int(np.sqrt(len(dictionary))))
h = emb(x)

tf.Tensor(
[[42 42 42 ...  0 42 57]
 [42 42 52 ... 42 57 56]
 [42 52 20 ... 57 56 30]
 ...
 [75 56 14 ... 36  7 56]
 [56 14 57 ...  7 56 82]
 [14 57 69 ... 56 82 54]], shape=(64, 127), dtype=int32) tf.Tensor(
[56 30 52 54  3 35 38  2 35 82  7 11 54  1 42 69 55 52 22 56 82  7 54 26
 69 65 54 57 55  2 42 35 42 79 52 61 35 54 57 69 55 30 35 64 14 69 42 55
 52 73 35 54 75 56 65 69  9 54 36  7 56 82 54  2], shape=(64,), dtype=int32)


In [ ]:
class Baseline(keras.layers.Layer):
    def call(self, inputs):
        return inputs[:,-1,...]

baseline = keras.Sequential([
    Baseline(),
    keras.layers.CategoryEncoding( num_tokens=len(dictionary), output_mode="one_hot")
])

baseline.compile(optimizer=keras.optimizers.Adam(),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))

baseline.evaluate(ds)

243/243 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4.4437


4.443704128265381

# Zadanie 

Zbudować model przewidujący następną literę w tekście napisów na podstawie atencji

- Dodać testowanie
- Dodać predykcje i generacje
- Zbadać czysty tekst i tekst z formatowanie (czas)
- Wyniki porównać z modelem odniesienia (baseline)

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

fname = "Star.Trek.Picard.S01E01.PROPER.WEBRip.x264-ION10.srt"
fname = "./Star.Trek.Picard.S01E01.PROPER.WEBRip.x264-ION10.srt"


with open(fname, "r", encoding="cp1250") as f:
    txt = f.read()
txts = txt.split("\n\n")


def parse(txt):
    subtxt = txt.split("\n")
    return subtxt[:2] + ["\n".join(subtxt[2:])]


df = pd.DataFrame([parse(txt) for txt in txts])
df.columns = ["index", "czas", "tekst"]
df = df.set_index("index")
df[["czas_start", "czas_stop"]] = df.czas.str.split("-->", expand=True)
df = df.drop("czas", axis=1)

## Zadanie  Dodatkowe kolumny
df.czas_start = pd.to_timedelta(df.czas_start.str.replace(",", ".")) / pd.to_timedelta(
    "1s"
)
df.czas_stop = pd.to_timedelta(df.czas_stop.str.replace(",", ".")) / pd.to_timedelta(
    "1s"
)
df["n"] = df.tekst.str.findall("\n").apply(len) + 1


df["ns"] = df["n"].astype(str)
df["multiline"] = df["n"] > 1
df["n"] = pd.Categorical(df["n"], categories=[1, 2])

df["l"] = df.tekst.str.len()
df["t"] = df.czas_stop - df.czas_start
df

korpus = ''.join(df['tekst'])
len(korpus)


15623

In [8]:
dictionary = list(set(korpus))
dictionary

korpus_tokeny = [dictionary.index(l) for l in korpus]

print(''.join(dictionary))

ódrPK3ZzŃśAęuĘl0BF,łN4Mx„2tODŁmźć
Soc:pWX?wJGHLYR”gI.- ean7UET!bżkŚÂfiń9ĆCŹj16hyŻ5są


In [ ]:
context=129
ds = keras.utils.timeseries_dataset_from_array(korpus_tokeny,None,sequence_length=context, batch_size=64, start_index=0, end_index=int(0.7*len(korpus)))

def splitfn(x):
    return x[...,:-1],x[...,-1]

ds = ds.map(splitfn)

In [10]:
for x,y in ds:
    break

dim = int(np.sqrt(len(dictionary)))
emb = keras.layers.Embedding(len(dictionary), dim)
h = emb(x)
print(h.shape)
h
at = keras.layers.MultiHeadAttention(4,dim)

h = at(h,h)
print(h.shape)

position_embedding = keras.layers.Embedding(context-1, dim)

pos = keras.ops.arange(0, context-1)
position_embedding(pos).shape
final_embed = emb(x)+keras.ops.expand_dims(position_embedding(pos),axis=0)
print(final_embed.shape)

(64, 128, 9)
(64, 128, 9)
(64, 128, 9)


In [12]:
import keras_nlp

encoder = keras_nlp.layers.TransformerEncoder(8,2,dropout=0.2)
h2 = encoder(h)
h2.shape

c:\aaasemestr9\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(64, 128, 9)

In [13]:
inp = keras.Input((None,))
h = keras.layers.Embedding(len(dictionary), dim)(inp)
#h = keras.layers.MultiHeadAttention(4,dim)(h,h)
keras_nlp.layers.TransformerEncoder(8,2,dropout=0.2)(h)
h = keras.layers.Lambda(lambda x: x[:,-1,...] )(h)
h = keras.layers.Dense(32, activation='relu')(h)
h = keras.layers.Dense(len(dictionary))(h)

model=keras.Model(inputs=inp, outputs=h)

model.compile(optimizer=keras.optimizers.Adam(),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
#model.predict(x)
h=model.fit(ds, epochs=4)

Epoch 1/4
169/169 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 3.9943
Epoch 2/4
169/169 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 3.3314
Epoch 3/4
169/169 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.1480
Epoch 4/4
169/169 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 3.0261


In [14]:
prompt='napisy '
prompt=r'00:41'
num_gen=200

tok_prompt = np.asarray([ dictionary.index(l) for l in prompt])
tok_prompt = np.expand_dims(tok_prompt, axis=0)

for _ in range(num_gen):

    #logits = model(tok_prompt) # batch
    logits = model.predict(tok_prompt, verbose=0) # batch
    next_token = keras.random.categorical(logits=logits,num_samples=1)

    tok_prompt = keras.ops.concatenate([tok_prompt, next_token], axis=1)


In [15]:
model.summary(expand_nested=True)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, None, 9)        │           756 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 9)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │         2,772 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,546 (45.10 KB)

 Trainable params: 3,848 (15.03 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,698 (30.07 KB)

In [16]:
print(''.join(np.take(dictionary, tok_prompt)[0]))

00:41ramęesnewoleiha zOżekymaieu wacnal 4rś.e  wcna .ćcwiegodopo jazy rę.Czlalale.To.I:gate nidoścząjrowo.r, nałe.IFukoz dć manieżoo ę„g,j witąm.Eesz ukem swo, pe  ć s ć,.cio dnasmetizzilobejOło.Tęi
ymrtNe
